# IN4080: obligatory assignment 4 - Part 1
 
The final mandatory assignment for IN4080 consists of two parts, which are distributed in two separate files. The first (this notebook) is about the development of dialogue systems, and the second (a different notebook) about machine translation. Both parts give a maximum of 10 points, and you are required to get at least 12/20 points to pass. Part 1 should be done on your local computer, as it relies on a speech interface that will not work on remote machines. For Part 2, using _Fox_ is preferable, at least for the fine-tuning task.

- We assume that you have read and are familiar with IFI’s requirements and guidelines for mandatory assignments, see [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-mandatory.html) and [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-guidelines.html).
- This is an individual assignment. You should not deliver joint submissions. 
- You may redeliver in Devilry before the deadline (__Friday, November 13 at 23:59__).
- Only the last delivery will be read! If you deliver more than one file, put them into a zip-archive. You don't have to include in your delivery the data files already provided for this assignment. 
- Submit two files and name your submissions _your\_username\_in4080\_mandatory\_4a_ / _your\_username\_in4080\_mandatory\_4b_

You should deliver a completed version of this Jupyter notebook, containing both your code and explanations about the steps you followed. We want to stress that simply submitting code is __not__ by itself sufficient to complete the assignment - we expect the notebook to also contain explanations of what you have implemented, along with motivations for the choices you made along the way. Preferably use whole sentences, and mathematical formulas if necessary. Explaining in your own words (using concepts we have covered through in the lectures) what you have implemented and reflecting on your solution is an important part of the learning process - take it seriously!

Regarding the use of LLMs (Claude, ChatGPT or similar): you are allowed to use them as 'sparring partner', for instance to clarify something you have not understood. However, you are __not__ allowed to use them to generate solutions (either in part or in full) to the assignment tasks. 

## Part 1 : Dialogue systems

Our objective in this part is to build a spoken conversational interface for a (simulated) elevator. 

### Basic setup

First, let's make sure that we have `ipywidgets` installed::

In [1]:
%pip install ipywidgets



Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install sounddevice openai-whisper pyttsx3

Note: you may need to restart the kernel to use updated packages.


The code for the simulated elevator is provided below. The elevator is displayed using simple widgets (where the current floor is shown in green). 

In [3]:
import ipywidgets as widgets
from IPython.display import display
import asyncio

import time, random, string, threading
from typing import List, Tuple, Dict, Set, Union
   
class BasicElevator:
    """Representation of an elevator. The elevator is simulated using a GUI"""
    
    def __init__(self, start_floor:int =1, nb_floors=10):
        """Initialised a new elevator, placed on the first floor"""
        
        # Current floor of the elevator
        self.cur_floor: int = start_floor
        self._moving: bool = False
        
        self._build_gui(nb_floors)

        # (Possibly empty) list of next floor stops to reach
        self.nextstops: List[int] = []

        display(self.gui)


    def move_to_floor(self, floor_number: int):
        """Move to a given floor (by adding it to a stack of floors to reach)"""
        
        if floor_number < 1 or floor_number > len(self.floors):
            raise RuntimeError("Floor number must be between 1 and %i"%len(self.floors))

        self.nextstops.append(floor_number)
        asyncio.create_task(self._trigger_movement())
        
        
    def stop(self):
        """Stops all movements of the elevator"""
        
        self.nextstops.clear()
        self._stop_generation = getattr(self, "_stop_generation", 0) + 1
        asyncio.create_task(self._trigger_movement())


    def _build_gui(self, nb_floors):
        """Creates the GUI for the elevator, with a status label and a visual representation
        of the floors, where the current floor is indicated in green."""

        # Displaying the current status of the elevator (still or going up or down)
        status_label = widgets.HTML("<b>Status</b>: ")
        self.status = widgets.Label("Still")
        status_box = widgets.HBox([status_label, self.status])

        # Displaying the floors on a vertical axis
        self.floors = []
        floor_layout = widgets.Layout(width='50px', height='30px', border='2px solid black',justify_content="center")
        for i in range(1, nb_floors+1):
            floor = widgets.Label(value=str(i), layout=floor_layout)
            floor.style = {"background":("white" if i!=self.cur_floor else "lightgreen")}
            self.floors.append(floor)

        # Create a vertical box container to hold the boxes
        vbox = widgets.VBox([status_box] + self.floors[::-1])
        self.gui = vbox

    async def _trigger_movement(self, speed=1.0):
        """Triggers the movement of the elevator towards the next stops in the list.
        The elevator moves one floor at a time, waiting 'speed' seconds between floors."""

        # If already moving, do nothing
        if self._moving:
            return
        
        self._moving = True
        while self.nextstops:
            floor_to_reach = self.nextstops[0]
            
            if self.cur_floor == floor_to_reach:
                self.nextstops.pop(0)
                continue
            
            if floor_to_reach > self.cur_floor:
                next_floor = self.cur_floor + 1
                self.status.value = "UP"
            else:
                next_floor = self.cur_floor - 1
                self.status.value = "DOWN"
            
            # Added sleep to simulate time taken to move between floors
            generation = getattr(self, "_stop_generation", 0)
            await asyncio.sleep(speed)
            if generation != getattr(self, "_stop_generation", 0):
                continue

            # Update UI safely
            self.floors[self.cur_floor - 1].style.background = "white"
            self.floors[next_floor - 1].style.background = "lightgreen"
            self.cur_floor = next_floor
             
        self.status.value = "Still"
        self._moving = False

The elevator can be easily controlled through the functions `move_to_floor` and `stop`:

In [4]:
elevator = BasicElevator()
elevator.move_to_floor(5)

We will now make our elevator controllable through a speech interface instead of using function calls.

Before we begin, you need to install `pyaudio` (for audio processing), `whisper` (for speech recognition), and `pyttsx3` (for speech synthesis):
```bash
%pip install pyaudio openai-whisper pyttsx3
```

Audio processing in Python can be quite tricky, as it works differently from OS to OS. In particular, if you are running on Linux and the TTS is not working, install the following packages on your machine: `sudo apt update && sudo apt install espeak ffmpeg libespeak1`. If you are on MacOS and TTS is not working, try installing portaudio using Homebrew: `brew install portaudio`.

If you run into issues with `pyaudio`, try using `sounddevice` instead: ```bash pip install sounddevice```.

_If you still run into issues with the audio processing, please let us know (ideally on Discourse), we have a number of fixes that can help_.

## Speech interface

The `TalkingElevator` class below extends the basic simulated elevator with speech input and output. 

Upon clicking on the recording button, speech is recorded from the user's microphone, and continues until the stop button is clicked. The speech recognition engine `Whisper` from OpenAI is then employed to transcribe the spoken input (either on GPU, if you have a GPU on your machine, or on CPU). The transcription result is then sent to the `process_input` function, which is responsible for determining the system response. 

We are going to focus on implementing this `process_input` method. Note this system reaction to new user inputs may comprise both verbal responses (to be uttered by the system through the `_say_to_user` method) and physical actions (through the `move_to_floor` and `stop` methods).

In [5]:
import threading, time
import numpy as np
from typing import List, Literal, Tuple, Dict, Set
import whisper, pyttsx3
import warnings

class TalkingElevator(BasicElevator):
    """Extension of the simulated elevator with a speech interface"""
    
    def __init__(self, audio_backend:Literal["pyaudio","sounddevice"]="pyaudio"):
        """Initialises the Talking Elevator, loading the ASR model and setting up the GUI.
        The audio_backend parameter specifies which library to use for audio recording
        (either 'pyaudio' or 'sounddevice')."""

        self.audio_backend = audio_backend
        print("Loading ASR model", end="...", flush=True)
        self.asr_engine = whisper.load_model("small.en")
        print("Done")
        
        # Initializing the GUI
        BasicElevator.__init__(self)

        # Starts the dialogue
        self.dialogue_history = []
        self._speech_lock = asyncio.Lock()
        self._speech_tasks = set()
        self._say_to_user("Hi, what can I do for you today?")
    

    def process_input(self, user_input: str, conf_score:float=1.0):
        """Processes the (transcribed) user input, and respond appropriately 
        (through a verbal response and possibly also an action, such as moving floors)"""

        self._add_to_dialogue_history(user_input, speaker="user", conf_score=conf_score)

        # Dummy response. Should be replaced by the actual dialogue behaviour
        self._say_to_user("Sorry, I don't understand you, pal")

    
    def _say_to_user(self, system_response: str):
        """Say something back to the user, and add the dialogue turn to the history. The 
        synthesis is done using the pyttsx3 library."""

        self._add_to_dialogue_history(system_response, speaker="elevator")
        
        def speak():
            tts_engine = pyttsx3.init()
            try:
                tts_engine.say(system_response)
                tts_engine.runAndWait()
            finally:
                tts_engine.stop()

        async def speak_in_order():
            async with self._speech_lock:
                await asyncio.to_thread(speak)

        def speech_finished(task):
            self._speech_tasks.discard(task)
            if not task.cancelled() and task.exception() is not None:
                warnings.warn(f"Speech synthesis failed: {task.exception()}", RuntimeWarning)

        task = asyncio.create_task(speak_in_order())
        self._speech_tasks.add(task)
        task.add_done_callback(speech_finished)


    def _add_to_dialogue_history(self, turn:str , speaker:str, conf_score:float=1.0):
        """Adds a new (user or system) turn to the dialogue history list, and displays it
         on the chat window displaying the turns"""

        self.dialogue_history.append({"speaker":speaker, "text":turn, 
                                      "conf_score":conf_score, "timesamp":time.time()})
        
        self.history_area.value += "&nbsp;<strong>%s</strong>:  %s"%(speaker.title(), turn)
        if conf_score < 1.0:
            self.history_area.value += " (%.2f)"%(conf_score)
        self.history_area.value += "<br>"
   
    
    def _build_gui(self, nb_floors):
        """GUI for the Talking elevator, comprising (beyond the simulated elevator from 
        BasicElevator) a chat window showing the dialogue turns, and buttons to record
        the user input. 
        The user should first click on the record button, then on stop when they have finished.
        Once the stop button is clicked, the audio is transcribed by Whisper, and finally 
        forwarded to the process_input function."""

        BasicElevator._build_gui(self, nb_floors)

        self.frames = []
        self.recording = False

        def record(chunk_size=1024):
            """Record audio chunks to a buffer."""
            if self.audio_backend == "pyaudio":
                import pyaudio
                p = pyaudio.PyAudio()
                try:
                    stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000,
                                    input=True, frames_per_buffer=chunk_size)
                    try:
                        while self.recording:
                            self.frames.append(stream.read(chunk_size))
                    finally:
                        stream.close()
                finally:
                    p.terminate()
            elif self.audio_backend == "sounddevice":
                import sounddevice
                with sounddevice.InputStream(samplerate=16000, channels=1, blocksize=chunk_size) as stream:
                    while self.recording:
                        data, overflowed = stream.read(chunk_size)
                        if overflowed:
                            warnings.warn("Audio input overflow occurred.", RuntimeWarning)
                        self.frames.append(data.copy())
            else:
                raise ValueError(f"Unsupported audio backend: {self.audio_backend}")

        def on_record_button_clicked(b):
            "Starts the recording"
            record_button.disabled = True
            stop_button.disabled = False
            audio_status.value = "Recording..."
            self.frames = []
            self.recording = True
            self._recording_task = asyncio.create_task(asyncio.to_thread(record))

        def on_stop_button_clicked(b):
            "stops the recording, runs Whisper, and forward the result to process_input"
            self.recording = False
            stop_button.disabled = True
            self._transcription_task = asyncio.create_task(finish_recording())
            self._transcription_task.add_done_callback(transcription_finished)

        def transcription_finished(task):
            if not task.cancelled() and task.exception() is not None:
                audio_status.value = f"Audio processing failed: {task.exception()}"
                warnings.warn(str(task.exception()), RuntimeWarning)

        async def finish_recording():
            try:
                await self._recording_task
                if not self.frames:
                    audio_status.value = "No audio recorded. Please try again."
                    return
                if self.audio_backend == "pyaudio":
                    audio_data = np.frombuffer(b"".join(self.frames), np.int16).astype(np.float32) / 32768.0
                else:
                    audio_data = np.concatenate(self.frames, axis=0).flatten()
                if audio_data.size == 0:
                    audio_status.value = "No audio recorded. Please try again."
                    return
                audio_status.value = "Transcribing..."
                output = await asyncio.to_thread(self.asr_engine.transcribe, audio_data, fp16=False)
                if not output["text"].strip() or not output["segments"]:
                    audio_status.value = "No speech recognized. Please try again."
                    return
                logprobs = [segment["avg_logprob"] for segment in output["segments"]]
                conf_score = float(np.exp(np.mean(logprobs)))
                if not np.isfinite(conf_score):
                    raise ValueError("Whisper returned an invalid confidence score.")
                conf_score = min(1.0, conf_score * 1.2)
                audio_status.value = "Done transcribing."
                self.process_input(output["text"], conf_score)
            finally:
                record_button.disabled = False

        # The record and stop buttons
        record_button = widgets.Button(icon="microphone")
        stop_button = widgets.Button(icon="stop", disabled=True)
        audio_status = widgets.Label("")
        record_button.on_click(on_record_button_clicked)
        stop_button.on_click(on_stop_button_clicked)
        
        # The chat area
        self.history_area = widgets.HTML(layout=widgets.Layout(width="600px", height="300px", 
                                                               border='1px solid black', overflow='scroll'))

        # The right side of the GUI
        right_side = widgets.VBox([widgets.Label(""), self.history_area, widgets.HBox([record_button, stop_button, audio_status])])
        self.gui = widgets.HBox([self.gui, right_side])


Let's give it a try:

In [6]:
elevator = TalkingElevator(audio_backend="sounddevice")

Loading ASR model...Done




**Note**: The current implementation reloads the TTS and ASR models every time, which means that you may run into a "CUDA: out of memory" error if you reinitialise the `TalkingElevator` many times. If this happens, simply restart the Python kernel, which will clear the memory on both the CPU and the GPU. 


### Intent recognition

We wish our talking elevator to support the following functionalities:
 
- If the user express a wish to go to floor $X$ (where $X$ is an integer value between 1 and 10), the elevator should go to that floor. The interface should allow for several ways to express a given intent, such as "_Please go to the $X$-th floor_" or "_Floor $X$, please_".
- The user requests can also be relative, for instance "_Go one floor up_".
- The elevator should provide _grounding_ feedback to the user. For instance, it should respond "_Ok, going to the $X$-th floor_" after a user request to move to $X$.  
- The elevator should handle misunderstandings and uncertainties, e.g. by requesting the user to repeat, or asking the user to confirm if the intent is uncertain (say, when its confidence score is lower than 0.5). 
- The elevator should also allow the user to ask where the office of a given employee is located. For instance, the user could ask "_where is Erik Velldal's office?_", and the elevator would provide a response such as "_The office of Erik Velldal is on the 4th floor. Do you wish to go there?_".  We provide you with the office numbers of a small set of IFI employees in the `OFFICES` dictionary (see below).
- The elevator should also be able to inform the user about the current floor (such as replying to "_Which floor are we on?_" or "_Are we on the 5th floor?_"). 
- Finally, if the user asks the elevator to stop (or if the user says "_no_" after a grounding feedback "_Ok, going to floor $X$._"), the elevator should stop, and ask for clarification regarding the actual user intent. 

To implement this conversational behaviour, we will rely on a classical NLU-based approach in which we will recognise the user _intent_, and then determine a response based on the recognised intent(s). 

__Task 1.1__ (1 point): You first need to define a list of user intents that cover the kinds of user inputs you expect to observe in this talking elevator, such as `RequestMoveToFloor` or `Confirm`. This is a design question, and there is no obvious right or wrong answer. Define below the intents you want to cover, along with an explanation and a few examples of user inputs for each.


In [ ]:
#RequestMoveToFloor

#The user wants to move to a specific floor.

#Examples:
#Go to the 5th floor
#Floor three please
#Take me to the seventh floor

#RequestRelativeMove
#The the user wants to move up or down from the current floor.
#Examples:
#Go one floor up.
#Move down two floors.
#Take me one floor higher.

#RequestCurrentFloor
#The user wants to know the current floor.
#Examples:
#Which floor are we on?
#What floor is this?
#Are we on the fifth floor?


#RequestOfficeLocation
#The user wants to know where someone's office is.
#Examples:
#Where is Erik Velldal's office?
#Which floor is Peter's office on?
#Take me to Anna's office.

#Stop
#The user wants to stop the elevator.
#Examples:
#Stop the elevator.
#Stop right now.
#Please stop.

#Confirm
#The user wants to confirm a previous request.
#Examples:
#Yes, please.
#That's correct.
#Yes, go ahead.

#Reject
#The user wants to reject or cancel a previous request.
#Examples:
#No, thanks.
#That's not correct.
#No, cancel that.

#OutOfCoverage
#The user says something that the elevator cannot handle.
#Examples:
#What's the weather today?
#Tell me a joke.
#What time is it?


<!-- Provide here the list of intent classes you have defined, together with an explanation and a few examples -->

__Task 1.2__ (1 point): We wish to build a classifier that can map any user input to a probability distribution over those intents, and start by creating a small, synthetic set of examples we will give to the classifier as few-shots. Make a list of about 50 user utterances, each labelled with an intent defined above. You can "make up" those utterances yourself, or ask someone else to come with alternative formulations if you lack inspiration.

In [8]:
labelled_utterances = [
    #RequestMoveToFloor
    ("Go to the fifth floor", "RequestMoveToFloor"),
    ("Floor three, please", "RequestMoveToFloor"),
    ("Take me to the seventh floor", "RequestMoveToFloor"),
    ("I want to go to the first floor", "RequestMoveToFloor"),
    ("Can you take me to floor four?", "RequestMoveToFloor"),
    ("Go to floor two", "RequestMoveToFloor"),
    ("Please take me to the sixth floor", "RequestMoveToFloor"),
    ("I need to go to floor eight", "RequestMoveToFloor"),
    ("Take me to the ground floor", "RequestMoveToFloor"),
    ("I want to go to floor ten", "RequestMoveToFloor"),

    #RequestRelativeMove
    ("Go one floor up", "RequestRelativeMove"),
    ("Move down two floors", "RequestRelativeMove"),
    ("Take me one floor higher", "RequestRelativeMove"),
    ("Take me one floor lower", "RequestRelativeMove"),
    ("Move up two floors", "RequestRelativeMove"),
    ("Go down one floor", "RequestRelativeMove"),
    ("Can we go up three floors?", "RequestRelativeMove"),
    ("Please move one floor down", "RequestRelativeMove"),

    #RequestCurrentFloor
    ("Which floor are we on?", "RequestCurrentFloor"),
    ("What floor is this?", "RequestCurrentFloor"),
    ("Where are we now?", "RequestCurrentFloor"),
    ("Can you tell me the current floor?", "RequestCurrentFloor"),
    ("Are we on the third floor?", "RequestCurrentFloor"),
    ("What is our current floor?", "RequestCurrentFloor"),

    #RequestOfficeLocation
    ("Where is Erik Velldal's office?", "RequestOfficeLocation"),
    ("Where can I find Erik Velldal?", "RequestOfficeLocation"),
    ("Which floor is Erik Velldal's office on?", "RequestOfficeLocation"),
    ("Can you tell me where Erik Velldal's office is?", "RequestOfficeLocation"),
    ("I am looking for Erik Velldal's office", "RequestOfficeLocation"),
    ("How can I get to Erik Velldal's office?", "RequestOfficeLocation"),

    #Stop
    ("Stop the elevator", "Stop"),
    ("Please stop", "Stop"),
    ("Stop here", "Stop"),
    ("Can you stop the elevator?", "Stop"),
    ("I want to stop now", "Stop"),

    #Confirm
    ("Yes, please", "Confirm"),
    ("Yes, that's correct", "Confirm"),
    ("That's right", "Confirm"),
    ("Sure", "Confirm"),
    ("Yes, go ahead", "Confirm"),

    #Reject
    ("No, that's wrong", "Reject"),
    ("No, thank you", "Reject"),
    ("That's not correct", "Reject"),
    ("No, I don't want that", "Reject"),
    ("No, please don't", "Reject"),

    #OutOfCoverage
    ("I have no comment", "OutOfCoverage"),
    ("I really like the IN4080 course", "OutOfCoverage"),
    ("What is the next floor?", "OutOfCoverage"),
    ("What was the previous floor?", "OutOfCoverage"),
    ("What is the weather today?", "OutOfCoverage")
]

from collections import Counter

print("Total utterances:", len(labelled_utterances))
print(Counter(intent for utterance, intent in labelled_utterances))



Total utterances: 50
Counter({'RequestMoveToFloor': 10, 'RequestRelativeMove': 8, 'RequestCurrentFloor': 6, 'RequestOfficeLocation': 6, 'Stop': 5, 'Confirm': 5, 'Reject': 5, 'OutOfCoverage': 5})


Conclusion 

The total utterances is 50

And this is fine.



We will now create  an intent classifier based on the labelled utterances you have defined. To make things simple, we will use [GLiClass](https://github.com/Knowledgator/GLiClass) as few-shot classifier. The basic idea is that you should load the pretrained model (for instance ``knowledgator/gliclass-multilang-mini``) with intent labels and your labelled examples. Check the GLiClass website for details. 

__Task 1.3__ (2 points): Implement `IntentClassifier` using GLiClass. If the number of labelled examples you have exceed the capability of GLiClass, simply use a fraction of your dataset. `get_intent_distrib` must return a dictionary containing every candidate intent and its probability, with probabilities summing to one.

In [9]:
%pip install gliclass

Note: you may need to restart the kernel to use updated packages.


In [10]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

Python: 3.14.7 | packaged by Anaconda, Inc. | (main, Aug 14 2026, 10:12:46) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.13.0+cu130
Transformers: 5.22.1


In [11]:
from typing import List, Tuple
import torch
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer

class IntentClassifier:
    def __init__(self, examples: List[Tuple], intent_labels: List[str]):
        """Load GLiClass and store a few labelled in-context examples."""

        #Store all unique intent labels and remove duplicates
        self.intent_labels = sorted(set(intent_labels))

        self.examples = []

        for label in self.intent_labels:

            #Select 3 examples for each intent 

            selected = [(text, intent) for text, intent in examples if intent == label][:3]

            for text,intent in selected : 


                self.examples.append({"text": text, "labels": [intent]})

        #Load the pretrained GLiClass model

        model_name = "knowledgator/gliclass-multilang-mini"


        #CUDA or CPU

        device = "cuda:0" if torch.cuda.is_available() else "cpu"

        #Load the pretrained model from Hugging Face
        model = GLiClassModel.from_pretrained(model_name)

        #Load the tokenizer to convert text into tokens
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        #Create the classification pipeline
        #Multi-label gives scores for different possible intents
        self.pipeline = ZeroShotClassificationPipeline(model, tokenizer, classification_type="multi-label", device=device)
        

    def get_intent_distrib(self, utterance: str) -> dict[str, float]:
        """Applies the model on the new utterance. The method should return a
        dictionary that maps each possible intent category to a probability."""

        # Predict intent scores
        results = self.pipeline(utterance, self.intent_labels, examples=self.examples, threshold=0.0)[0]

        # Store scores for all intents
        scores = {label: 0.0 for label in self.intent_labels}
        for result in results:
            scores[result["label"]] = float(result["score"])

        # Normalize probabilities
        total = sum(scores.values())
        if total > 0:

            
            return {label: score / total for label, score in scores.items()}
        
        return {label: 1.0 / len(self.intent_labels) for label in self.intent_labels}

intent_labels = sorted(set(l for _, l in labelled_utterances))


intent_labels = sorted(set(l for _, l in labelled_utterances))
classifier = IntentClassifier(examples=labelled_utterances, intent_labels=intent_labels)

#Test
utterance = "Could you take me to floor nine?"
result = classifier.get_intent_distrib(utterance)

for intent, probability in result.items():
    print(f"{intent}: {probability:.4f}")

print("Total:", sum(result.values()))
print("Predicted:", max(result, key=result.get))


Loading weights:   0%|          | 0/227 [00:00<?, ?it/s]

[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
100%|██████████| 1/1 [00:00<00:00,  1.18it/s]

Confirm: 0.0303
OutOfCoverage: 0.1410
Reject: 0.1755
RequestCurrentFloor: 0.0588
RequestMoveToFloor: 0.4445
RequestOfficeLocation: 0.0443
RequestRelativeMove: 0.0257
Stop: 0.0800
Total: 1.0
Predicted: RequestMoveToFloor


In [12]:
#Test 2 
test_sentences = [
    "Go up two floors",
    "Which floor are we on?",
    "Where is Erik Velldal's office?",
    "Stop the elevator",
    "Yes, please",
    "No, that's wrong",
    "What is the weather today?"
]

for sentence in test_sentences:
    result = classifier.get_intent_distrib(sentence)
    predicted = max(result, key=result.get)
    print(f"{sentence} -> {predicted} ({result[predicted]:.4f})")




test_cases = [
    ("Go to the fifth floor", "RequestMoveToFloor"),
    ("Go up two floors", "RequestRelativeMove"),
    ("Which floor are we on?", "RequestCurrentFloor"),
    ("Where is Erik Velldal's office?", "RequestOfficeLocation"),
    ("Stop the elevator", "Stop"),
    ("Yes, please", "Confirm"),
    ("No, that's wrong", "Reject"),
    ("What is the weather today?", "OutOfCoverage")
]

correct = 0

for sentence, expected in test_cases:
    result = classifier.get_intent_distrib(sentence)
    predicted = max(result, key=result.get)
    confidence = result[predicted]

    if predicted == expected:
        correct += 1

    print(f"{sentence}")
    print(f"Expected: {expected}")
    print(f"Predicted: {predicted}")
    print(f"Confidence: {confidence:.4f}")
    print(f"Total probability: {sum(result.values()):.4f}")
    print()

print(f"Accuracy: {correct}/{len(test_cases)}")


100%|██████████| 1/1 [00:00<00:00,  4.39it/s]


Go up two floors -> RequestMoveToFloor (0.2819)


100%|██████████| 1/1 [00:00<00:00, 11.46it/s]


Which floor are we on? -> RequestCurrentFloor (0.2679)


100%|██████████| 1/1 [00:00<00:00,  4.22it/s]


Where is Erik Velldal's office? -> RequestOfficeLocation (0.4820)


... (20 outputs left out)

100%|██████████| 1/1 [00:00<00:00, 34.30it/s]


No, that's wrong
Expected: Reject
Predicted: Reject
Confidence: 0.5943
Total probability: 1.0000



100%|██████████| 1/1 [00:00<00:00, 42.59it/s]

What is the weather today?
Expected: OutOfCoverage
Predicted: OutOfCoverage
Confidence: 0.3153
Total probability: 1.0000

Accuracy: 7/8


Conclusion



Tested the classifier with 8 different sentences.

The model correctly predicted 7 out of 8 intents, with an accuracy of 87.5%.

However, "Go up two floors" was predicted as RequestMoveToFloor instead of RequestRelativeMove.

The model works for most of the test sentences, but it still makes some mistakes.


You can then test your classifier as follows:

In [13]:
classifier.get_intent_distrib("Take me to the fifth floor, please")

100%|██████████| 1/1 [00:00<00:00, 32.69it/s]


{'Confirm': 0.12585492756691274,
 'OutOfCoverage': 0.04836917882615945,
 'Reject': 0.03299814795774492,
 'RequestCurrentFloor': 0.03155661312572951,
 'RequestMoveToFloor': 0.6219699454192574,
 'RequestOfficeLocation': 0.046983011128244906,
 'RequestRelativeMove': 0.025975567191866958,
 'Stop': 0.06629260878408405}

Conclusion

The model correctly predicted RequestMoveToFloor with a probability of 0.6220.

Since we don't have any test data, we cannot really conduct an evaluation of the classification performance, but this step would be of course strongly advised when developing a real system.

### Slot filling

In addition to the intents themselves, we also wish to detect some slots, such as floor numbers or person names. For this step, we will not use a data-driven model, but rather rely on an old-fashioned, rule-based approach:
- For floor numbers, we will rely on string matching (with regular expressions or basic string search) that detect patterns such as "X floor" (where X is [first,second, third, fourth, fifth, sixth, seventh, eighth, ninth, tenth]) or "floor X" (where X is between 1 and 10).
- For person names, we have a predefined list of person names to detect (employees at IFI), and we should simply search for their occurrence in the user input. The simplest implementation is to just for look for exact occurrences. However, since speech recognition will often struggle to recognize foreign person names, an even better approach would be to search for names that are phonetically close (you can use the `jellyfish` library for this).

The results of the slot filling should be a dictionary mapping slot names to a canonical form of the slot value. For instance, if the utterance contains the expression "ninth floor", the resulting slot dictionary should be `{"floor_number":9}`. Similarly, the `employee_name` slot should be a name present in `OFFICES` dictionary. 

__Task 1.4__ (2 points): Implement the method `fill_slots` that will detect the occurrence of those slots in the user input.<br>
(+ 1 bonus point if you implement a fuzzy matching strategy to find person names that are phonetically close)

In [14]:

# Floor numbers for a subset of the IFI employees
OFFICES = {'Adín Ramírez Rivera': 4, 'Andreas Austeng': 4, 'Anne H Schistad Solberg': 4, 
           'Arild Torolv Søetorp Waaler': 9, 'Audun Jøsang': 9, 'Birthe Soppe': 4, 'Carsten Griwodz': 4,
           'Dag Sjøberg': 9, 'Dag Trygve Eckhoff Wisland': 5, 'Einar Broch Johnsen': 8, 
           'Eric Bartley Jul': 10, 'Erik Velldal': 4, 'Henrik Skaug Sætra': 7, 'Ingrid Chieh Yu': 8,
           'Jørn Anders Braa': 6, 'Kristin Bråthen': 4, 'Kyrre Glette': 4, 'Lars Groth': 6, 
           'Lilja Øvrelid': 4, 'Maja Van Der Velden': 7, 'Martin Giese': 9, 'Michael Welzl': 5, 
           'Miria Grisot': 6, 'Nils Gruschka': 9, 'Olaf Owe': 9, 'Ole Christian Lingjærde': 4, 
           'Ole Hanseth': 6, 'Paulo Ferreira': 10, 'Philipp Dominik Häfliger': 5, 'Philipp Häfliger': 5, 
           'Roman Vitenberg': 4, 'Silvia Lizeth Tapia Tarifa': 8, 'Stephan Oepen': 4, 
           'Sundeep Sahay': 6, 'Thomas Peter Plagemann': 4, 'Tone Bratteteig': 7, 'Torbjørn Rognes': 8, 
           'Truls Erikson': 6, 'Viktoria Stray': 10, 'Yngvar Berg': 5, 'Yves Scherrer': 4, 
           'Özgü Mira Alay-Erduran': 4}

#My own code

import re
from typing import Dict, Union

def fill_slots(user_input:str) -> Dict[str, Union[int, str]]:
    """Extracts the set of slots detected in the user inputs. More precisely, the method
    should detect both floor numbers and person names, and return a dictionary mapping slot 
    names (in this case either `floor_number` or `employee_name`) to its corresponding
    value, in canonical form (integer for the floor number, string for the employee name)"""

    slots = {}

    #Convert floor numbers and ordinal words to integers
    floor_numbers = { "one": 1, "first": 1, "two": 2, "second": 2, "three": 3, "third": 3, "four": 4, "fourth": 4, "five": 5, "fifth": 5, "six": 6, "sixth": 6, "seven": 7, "seventh": 7, "eight": 8, "eighth": 8, "nine": 9, "ninth": 9, "ten": 10, "tenth": 10 }


    #Add numerical representations from 1 to 10

    for i in range(1,11):
        floor_numbers[str(i)] = i 

    #Add ordinal numbers such as 1st, 2nd, 3rd and 4th
    suffixes = {1: "st", 2: "nd", 3: "rd"}
    for i in range(1, 11):
        suffix = suffixes.get(i, "th")
        floor_numbers[f"{i}{suffix}"] = i

    #Create a regex pattern for floor numbers
    number_pattern = "|".join(floor_numbers.keys())
    pattern = rf"\b({number_pattern})\s+floor\b|\bfloor\s+({number_pattern})\b"

    # Search for floor numbers in the user input
    match = re.search(pattern, user_input, re.IGNORECASE)
    if match:
        number = match.group(1) or match.group(2)
        slots["floor_number"] = floor_numbers[number.lower()]

    # Search for employee names in the OFFICES dictionary
    for name in sorted(OFFICES, key=len, reverse=True):
        if re.search(rf"(?<!\w){re.escape(name)}(?!\w)", user_input, re.IGNORECASE):
            slots["employee_name"] = name
            break

    return slots



# Test slot filling
test_sentences = [
    "Take me to the ninth floor",
    "Go to floor 5",
    "Please go to the 3rd floor",
    "Where is Erik Velldal's office?",
    "Take me to the fifth floor to meet Erik Velldal",
    "Stop the elevator",
    "What is the weather today?"
]

for sentence in test_sentences:
    print(sentence, "->", fill_slots(sentence))
    


Take me to the ninth floor -> {'floor_number': 9}
Go to floor 5 -> {'floor_number': 5}
Please go to the 3rd floor -> {'floor_number': 3}
Where is Erik Velldal's office? -> {'employee_name': 'Erik Velldal'}
Take me to the fifth floor to meet Erik Velldal -> {'floor_number': 5, 'employee_name': 'Erik Velldal'}
Stop the elevator -> {}
What is the weather today? -> {}


Conclusion

Tested the slot filling function with 7 different sentences.

The function correctly extracted floor numbers and employee names.

It can also extract both slots from the same sentence.

All 7 test cases worked correctly.

### Response selection

The next step is to implement the response selection mechanism. The response will depend on various factors:
- the inferred user intents from the user utterance
- the detected slot values in the user utterance (if any)
- the current floor
- the list of next floor stops that are yet to be reached
- the dialogue history (as a list of dialogue turns).

The response may consist of verbal responses (enacted by calls to `_say_to_user`) but also physical actions, represented by calls to either `move_to_floor` or `stop`. 

__Task 1.5__ (3 points): Implement the method `_respond`, which is responsible for selecting and executing those responses. The responses should satisfy the aforementioned conversational criteria (provide grounding feedback, use confirmations and clarification requests etc.). This method will consist in practice of many _if...then...else_ blocks. 

In [15]:
def _respond(self, intent_distrib: Dict[str, float], slots: Dict[str, Union[int, str]]) :
    """Given a probability distribution over possible intents, an a (possibly empty) list
    of detected slots in the user input, decide how to react. The method should lead
    to calls to both physical actions (move_to_floor, stop) and dialogue responses 
    (via _say_to_user)."""

    # Find the most likely intent and its probability
    intent = max(intent_distrib, key=intent_distrib.get)
    confidence = intent_distrib[intent]

    #Check whether the elevator is waiting for confirmation
    pending_floor = getattr(self, "_pending_floor", None)

    #Stop the elevator and cancel pending requests
    if intent == "Stop" or intent == "Reject":
        self._pending_floor = None
        self.stop()
        self._say_to_user("Okay, stopped. Where would you like to go?")
        return

    #Handle confirmation of a previous request
    if intent == "Confirm":
        if pending_floor is not None:
            self._pending_floor = None
            self._say_to_user(f"Okay, going to floor {pending_floor}.")
            self.move_to_floor(pending_floor)
        else:
            self._say_to_user("What would you like me to do?")
        return

    #Cancel an old request if the user gives a new command
    self._pending_floor = None

    #Handle requests outside the supported functionality
    if intent == "OutOfCoverage":
        self._say_to_user("Sorry, I cannot help with that request.")
        return

    # Ask for clarification when the intent is uncertain
    if confidence < 0.5:
        self._say_to_user("Sorry, I am not sure what you mean. Could you repeat?")
        return

    #Request to move to a specific floor
    if intent == "RequestMoveToFloor":
        floor = slots.get("floor_number")
        if floor is None:
            self._say_to_user("Which floor would you like to go to?")
        elif not 1 <= floor <= 10:
            self._say_to_user("Please choose a floor between 1 and 10.")
        elif floor == self.cur_floor:
            self._say_to_user(f"We are already on floor {floor}.")
        else:
            self._say_to_user(f"Okay, going to floor {floor}.")
            self.move_to_floor(floor)

    #Request to move up or down relative to the current floor
    elif intent == "RequestRelativeMove":
        text = getattr(self, "_last_user_input", "").lower()
        match = re.search(r"\b(one|two|three|four|five|six|seven|eight|nine|ten|\d+)\s+floors?\b", text)
        numbers = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10}
        steps = numbers.get(match.group(1), 1) if match and not match.group(1).isdigit() else int(match.group(1)) if match else 1

        if re.search(r"\b(up|higher)\b", text):
            floor = self.cur_floor + steps
        elif re.search(r"\b(down|lower)\b", text):
            floor = self.cur_floor - steps
        else:
            self._say_to_user("Would you like to go up or down?")
            return

        if 1 <= floor <= 10:
            self._say_to_user(f"Okay, going to floor {floor}.")
            self.move_to_floor(floor)
        else:
            self._say_to_user("Sorry, that floor is outside the building.")

    #Request information about the current floor
    elif intent == "RequestCurrentFloor":
        self._say_to_user(f"We are currently on floor {self.cur_floor}.")

    #Request information about an employee's office
    elif intent == "RequestOfficeLocation":
        name = slots.get("employee_name")
        if name is None:
            self._say_to_user("Whose office are you looking for?")
        elif name in OFFICES:
            floor = OFFICES[name]
            self._pending_floor = floor
            self._say_to_user(f"The office of {name} is on floor {floor}. Do you wish to go there?")
        else:
            self._say_to_user("Sorry, I could not find that employee.")

    #Handle any other unrecognized situation
    else:
        self._say_to_user("Sorry, could you clarify your request?")

setattr(TalkingElevator, "_respond", _respond)

#Result

print("Current floor:", elevator.cur_floor)
print("Move method:", hasattr(elevator, "move_to_floor"))
print("Stop method:", hasattr(elevator, "stop"))
print("Response method:", hasattr(elevator, "_respond"))




Current floor: 1
Move method: True
Stop method: True
Response method: True


Conclusion

The current floor is 1.
All three methods are available.

In [16]:

# Test 3
elevator._respond({"RequestMoveToFloor": 1.0}, {"floor_number": 5})
print(elevator.dialogue_history[-1]["text"])

# Test 4
elevator._respond({"Stop": 1.0}, {})
print(elevator.dialogue_history[-1]["text"])

# Test 5
elevator._respond({"RequestOfficeLocation": 1.0}, {"employee_name": "Erik Velldal"})
print(elevator.dialogue_history[-1]["text"])

elevator._respond({"Confirm": 1.0}, {})
print(elevator.dialogue_history[-1]["text"])


Okay, going to floor 5.
Okay, stopped. Where would you like to go?
The office of Erik Velldal is on floor 4. Do you wish to go there?
Okay, going to floor 4.


Conclusion

Tested moving, stopping and finding an office.

The elevator responded correctly to all requests.

It also asked for confirmation before going to floor 4.




### Putting it all together

The last step is to implement the `process_input` method in the `TalkingElevator` class. The method should rely on the intent recognition, slot filling and response selection mechanism (which you have implemented in the previous steps) to react to a given user input.

**Task 1.6** (1 point): Implement the `process_input` method:

In [17]:
def process_input(self, user_input: str, conf_score:float=1.0):
    """Processes the (transcribed) user input, and respond appropriately 
    (through a verbal response and possibly also an action, such as moving floors).
    The method should rely on the intent classifier, slot-filling function, and
    response selection function."""

    self._add_to_dialogue_history(user_input, speaker="user", conf_score=conf_score)
    # Store the user input for relative movement requests
    self._last_user_input = user_input

    # Check whether the speech recognition is reliable
    if conf_score < 0.5:
        self._say_to_user("Sorry, I did not understand. Could you repeat?")
        return

    # Step 1: Recognize the user's intent
    intent_distrib = classifier.get_intent_distrib(user_input)

    # Step 2: Extract floor numbers and employee names
    slots = fill_slots(user_input)

    # Step 3: Select and execute the response
    self._respond(intent_distrib, slots)

setattr(TalkingElevator, "process_input", process_input)


#Test
elevator.process_input("Take me to the fifth floor")
elevator.process_input("Which floor are we on?")
elevator.process_input("Where is Erik Velldal's office?")
elevator.process_input("Yes, please")
elevator.process_input("Go up two floors")
elevator.process_input("Stop the elevator")

100%|██████████| 1/1 [00:00<00:00, 45.66it/s]


We are now ready to test our talking elevator: 


In [19]:
elevator = TalkingElevator()

Loading ASR model...Done


Your talking elevator will mostly likely not function properly right from the start. Identify what works and what doesn't and correct the code you have developed in Tasks 1.1 - 1.6 until your system meets the specifications we have outlined. 

In [20]:
# Test
elevator.process_input("Take me to the fifth floor")
elevator.process_input("Which floor are we on?")
elevator.process_input("Where is Erik Velldal's office?")
elevator.process_input("Yes, please")

100%|██████████| 1/1 [00:00<00:00, 47.52it/s]
